In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import galsim

from shape_vs_intensity import config as C
from shape_vs_intensity.plotutils import use_style

use_style()

In [ ]:
# Create pupil grid
noll_indices = np.arange(1, C.JMAX + 1)
grid = np.linspace(-1.01, 1.01, C.NPIX * 10)
u, v = np.meshgrid(grid, grid)

# Create Zernike basis
zk = galsim.zernike.zernikeBasis(max(noll_indices), u, v, R_inner=C.EPS)[
    min(noll_indices) :
]

# Mask outside annulus
r = np.sqrt(u**2 + v**2)
mask = (r >= C.EPS) & (r <= 1)

# Apply mask
zk *= mask

In [ ]:
# Create title dictionary
titles = {
    1: "Piston",
    2: "Tilt",
    3: "Tilt",
    4: "Defocus",
    5: "Astigmatism",
    6: "Astigmatism",
    7: "Coma",
    8: "Coma",
    9: "Trefoil",
    10: "Trefoil",
    11: "Spherical",
    12: "2nd Astigmatism",
    13: "2nd Astigmatism",
    14: "Quadrafoil",
    15: "Quadrafoil",
    16: "2nd Coma",
    17: "2nd Coma",
    18: "2nd Trefoil",
    19: "2nd Trefoil",
    20: "Pentafoil",
    21: "Pentafoil",
    22: "2nd Spherical",
}

In [ ]:
# Create the figure
fig = plt.Figure(figsize=(7.5, 9), dpi=120)

# Use Josh's code to setup pyramid
jmin = min(noll_indices)
jmax = max(noll_indices)
jdict = {x: y for x, y in zip(noll_indices, range(len(noll_indices)))}
nmax, _ = galsim.zernike.noll_to_zern(jmax)
nmin, _ = galsim.zernike.noll_to_zern(jmin)

nrow = nmax - nmin + 1
ncol = nmax + 1
gridspec = GridSpec(nrow, ncol)


def shift(pos, amt):
    return [pos.x0 + amt, pos.y0, pos.width, pos.height]


def shiftAxes(axes, amt):
    for ax in axes:
        ax.set_position(shift(ax.get_position(), amt))


axes = {}
shiftLeft = []
shiftRight = []
for j in noll_indices:
    n, m = galsim.zernike.noll_to_zern(j)
    if n % 2 == 0:
        row, col = n - nmin, m // 2 + ncol // 2
    else:
        row, col = n - nmin, (m - 1) // 2 + ncol // 2
    subplotspec = gridspec.new_subplotspec((row, col))
    axes[j] = fig.add_subplot(subplotspec)
    axes[j].set_aspect("equal")
    if nmax % 2 == 0 and (nmax - n) % 2 == 1:
        shiftRight.append(axes[j])
    if nmax % 2 == 1 and (nmax - n) % 2 == 1:
        shiftLeft.append(axes[j])

# Loop through and plot polynomials
for j, ax in axes.items():
    if j not in noll_indices:
        continue
    ax.set_title(f"{j}\n{titles[j]}", fontsize=10)
    ax.imshow(zk[jdict[j]], origin="lower", cmap="bwr")

    # Make sure colors are symmetric about 0
    img = ax.get_images()[0]
    vmax = np.abs(img.get_clim()).max()
    img.set_clim(-vmax, +vmax)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.set_axis_off()

fig.tight_layout()

# Assume we always have Z4 and Z5?
amt = 0.5 * (axes[4].get_position().x0 - axes[5].get_position().x0)
shiftAxes(shiftLeft, -amt)
shiftAxes(shiftRight, amt)

fig.savefig(C.FIGDIR / "zernike_pyramid.pdf")

In [ ]:
# Zernikes 4--28 in a triangle ordered by radial order n and angular frequency |m|.
triangle_indices = np.arange(4, 29)
triangle_titles = {
    4: "Defocus",
    5: "Astigmatism",
    6: "Astigmatism",
    7: "Coma",
    8: "Coma",
    9: "Trefoil",
    10: "Trefoil",
    11: "Spherical",
    12: "2nd Astig.",
    13: "2nd Astig.",
    14: "Quadrafoil",
    15: "Quadrafoil",
    16: "2nd Coma",
    17: "2nd Coma",
    18: "2nd Trefoil",
    19: "2nd Trefoil",
    20: "Pentafoil",
    21: "Pentafoil",
    22: "2nd Spherical",
    23: "3rd Astig.",
    24: "3rd Astig.",
    25: "2nd Quad.",
    26: "2nd Quad.",
    27: "Hexafoil",
    28: "Hexafoil",
}

triangle_basis = galsim.zernike.zernikeBasis(
    max(triangle_indices), u, v, R_inner=C.EPS
)
triangle_zk = {j: triangle_basis[j] * mask for j in triangle_indices}


def triangle_slot(m):
    """Map signed azimuthal order to compact left-aligned m slots."""
    if m == 0:
        return 0
    return 2 * abs(m) - 1 + int(m > 0)


rows = {}
for j in triangle_indices:
    n, m = galsim.zernike.noll_to_zern(j)
    rows.setdefault(n, []).append((j, m))
row_slots = {
    n: sorted(triangle_slot(m) for _, m in row)
    for n, row in rows.items()
}
rows = {n: sorted(row, key=lambda row: row[0]) for n, row in rows.items()}

fig = plt.figure(figsize=(14.0, 6.3), dpi=150)
panel_width = 0.074
panel_height = panel_width * fig.get_figwidth() / fig.get_figheight()
xgap = 0.001
x0 = 0.008
y_positions = np.linspace(0.70, 0.04, len(rows))

for row, n in zip(y_positions, sorted(rows)):
    for slot, (j, m) in zip(row_slots[n], rows[n]):
        ax = fig.add_axes(
            [x0 + slot * (panel_width + xgap), row, panel_width, panel_height]
        )
        img = ax.imshow(triangle_zk[j], origin="lower", cmap="bwr")
        vmax = np.abs(triangle_zk[j]).max()
        img.set_clim(-vmax, vmax)

        ax.set_title(
            f"Z{j}\n{triangle_titles[j]}",
            fontsize=10.5,
            pad=0,
        )
        ax.set(xticks=[], yticks=[])
        ax.set_aspect("equal")
        ax.set_axis_off()

fig.savefig(C.FIGDIR / "zernike_triangle.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(7, 4), dpi=150)


def plot_zk(j, ax):
    # Plot image
    ax.imshow(zk[j - 1], origin="lower", cmap="bwr")

    # Make sure colors are symmetric about 0
    img = ax.get_images()[0]
    vmax = np.abs(img.get_clim()).max()
    img.set_clim(-vmax, +vmax)

    # Set title
    ax.text(
        0.5,
        1.05,
        f"{titles[j]}",
        ha="center",
        va="bottom",
        transform=ax.transAxes,
        fontsize=8,
    )


# nu = 0
plot_zk(4, axes[0, 0])
plot_zk(2, axes[0, 1])
plot_zk(6, axes[0, 2])
plot_zk(10, axes[0, 3])
plot_zk(14, axes[0, 4])
plot_zk(20, axes[0, 5])

# nu = 1
plot_zk(11, axes[1, 0])
plot_zk(8, axes[1, 1])
plot_zk(12, axes[1, 2])
plot_zk(18, axes[1, 3])

# nu = 2
plot_zk(22, axes[2, 0])
plot_zk(16, axes[2, 1])

for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.set(xticks=[], yticks=[], aspect="equal")

for i in range(len(axes)):
    axes[i, 0].set_ylabel(
        i,
        rotation=0,
        labelpad=18,
        fontsize=12,
        ha="center",
        va="center",
    )

for i in range(len(axes[0])):
    axes[0, i].set_title(i, pad=25, fontsize=12)

fig.text(0.5, 1.04, "$m$", fontsize=16, ha="center", va="center")
fig.text(0.035, 0.5, "$\\nu$", fontsize=16, ha="center", va="center")

fig.savefig(C.FIGDIR / "zernike_nu_table.pdf", bbox_inches="tight")

In [ ]:
def shared_color_limit(values, percentile=97.5, zero_tol=1e-10):
    zero_atol = zero_tol * np.nanmax(np.abs(values))
    nonzero = np.abs(values[np.abs(values) >= zero_atol])
    if len(nonzero) == 0:
        return 1.0
    return np.nanpercentile(nonzero, percentile)


def shared_gradient_scale(x_values, y_values, percentile=97.5, target_length=0.28):
    magnitude = np.hypot(x_values, y_values)
    nonzero = magnitude[magnitude > 0]
    if len(nonzero) == 0:
        return 1.0
    return np.nanpercentile(nonzero, percentile) / target_length


def plot_zernike_pyramid(
    values,
    out,
    vmax=None,
    zero_tol=1e-10,
    quiver=None,
    quiver_scale=None,
    quiver_nr=3,
    quiver_ntheta=16,
    min_quiver_length=0.0,
):
    fig = plt.Figure(figsize=(7.5, 9), dpi=120)
    zero_atol = zero_tol * np.nanmax(np.abs(values))
    if vmax is None:
        vmax = shared_color_limit(values, zero_tol=zero_tol)
    quiver_zero_atol = 0.0
    if quiver is not None:
        quiver_mag_all = np.hypot(quiver[0], quiver[1])
        quiver_zero_atol = zero_tol * np.nanmax(quiver_mag_all)
    extent = [grid[0], grid[-1], grid[0], grid[-1]]
    q_r = np.linspace(C.EPS + 0.08 * (1 - C.EPS), 0.92, quiver_nr)
    q_theta = np.linspace(0, 2 * np.pi, quiver_ntheta, endpoint=False)
    q_rr, q_tt = np.meshgrid(q_r, q_theta, indexing="ij")
    q_tt += (np.arange(quiver_nr)[:, None] % 2) * np.pi / quiver_ntheta
    q_u = q_rr * np.cos(q_tt)
    q_v = q_rr * np.sin(q_tt)

    jmin = min(noll_indices)
    jmax = max(noll_indices)
    jdict = {x: y for x, y in zip(noll_indices, range(len(noll_indices)))}
    nmax, _ = galsim.zernike.noll_to_zern(jmax)
    nmin, _ = galsim.zernike.noll_to_zern(jmin)

    nrow = nmax - nmin + 1
    ncol = nmax + 1
    gridspec = GridSpec(nrow, ncol)

    def shift(pos, amt):
        return [pos.x0 + amt, pos.y0, pos.width, pos.height]

    def shift_axes(axes, amt):
        for ax in axes:
            ax.set_position(shift(ax.get_position(), amt))

    axes = {}
    shiftLeft = []
    shiftRight = []
    for j in noll_indices:
        n, m = galsim.zernike.noll_to_zern(j)
        if n % 2 == 0:
            row, col = n - nmin, m // 2 + ncol // 2
        else:
            row, col = n - nmin, (m - 1) // 2 + ncol // 2
        subplotspec = gridspec.new_subplotspec((row, col))
        axes[j] = fig.add_subplot(subplotspec)
        axes[j].set_aspect("equal")
        if nmax % 2 == 0 and (nmax - n) % 2 == 1:
            shiftRight.append(axes[j])
        if nmax % 2 == 1 and (nmax - n) % 2 == 1:
            shiftLeft.append(axes[j])

    for j, ax in axes.items():
        panel = values[jdict[j]].copy()
        panel[np.abs(panel) < zero_atol] = 0.0

        ax.set_title(f"{j}\n{titles[j]}", fontsize=10)
        img = ax.imshow(panel, origin="lower", cmap="bwr", extent=extent)

        img.set_clim(-vmax, vmax)

        if quiver is not None:
            coeff = np.zeros(max(noll_indices) + 1)
            coeff[j] = 1.0
            zernike = galsim.zernike.Zernike(coeff, R_inner=C.EPS)
            qx, qy = zernike.evalCartesianGrad(q_u, q_v)
            qx *= -1
            qy *= -1
            q_mag = np.hypot(qx, qy)
            q_plot_mask = q_mag > quiver_zero_atol
            if np.any(q_plot_mask):
                qx_plot = qx.copy()
                qy_plot = qy.copy()
                scale = quiver_scale
                if min_quiver_length > 0 and quiver_scale is not None:
                    display_length = q_mag[q_plot_mask] / quiver_scale
                    length_boost = np.maximum(min_quiver_length / display_length, 1.0)
                    qx_plot[q_plot_mask] = qx[q_plot_mask] * length_boost
                    qy_plot[q_plot_mask] = qy[q_plot_mask] * length_boost
                ax.quiver(
                    q_u[q_plot_mask],
                    q_v[q_plot_mask],
                    qx_plot[q_plot_mask],
                    qy_plot[q_plot_mask],
                    angles="xy",
                    scale_units="xy",
                    scale=scale,
                    width=0.009,
                    headwidth=3.0,
                    headlength=4.0,
                    headaxislength=3.4,
                    pivot="mid",
                    color="0.15",
                    alpha=0.78,
                )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")
        ax.set_axis_off()

    fig.tight_layout()

    amt = 0.5 * (axes[4].get_position().x0 - axes[5].get_position().x0)
    shift_axes(shiftLeft, -amt)
    shift_axes(shiftRight, amt)

    fig.savefig(out)
    plt.close(fig)


grad_zk = galsim.zernike.zernikeGradBases(
    max(noll_indices), u, v, R_inner=C.EPS
)[:, min(noll_indices) :]
zk_dx = -grad_zk[0] * mask
zk_dy = -grad_zk[1] * mask

zk_laplacian = np.empty_like(zk_dx)
for i, j in enumerate(noll_indices):
    coeff = np.zeros(max(noll_indices) + 1)
    coeff[j] = 1.0
    zernike = galsim.zernike.Zernike(coeff, R_inner=C.EPS)
    zk_laplacian[i] = zernike.laplacian(u, v) * mask

laplacian_vmax = shared_color_limit(zk_laplacian)
gradient_quiver_scale = shared_gradient_scale(zk_dx, zk_dy)

plot_zernike_pyramid(
    zk_laplacian,
    C.FIGDIR / "zernike_curvature_and_displacements.pdf",
    vmax=laplacian_vmax,
    quiver=(zk_dx, zk_dy),
    quiver_scale=gradient_quiver_scale,
    min_quiver_length=0.10,
)